In [9]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


##### /!\ Il y a eu un pb avec la bdd de 2020, il faut màj l'interprétation qui suit les graphes (erif que ce sont les bons chiffres par ex)

In [10]:
import importlib
from pathlib import Path
import pandas as pd
import src.collect_licences as cl

importlib.reload(cl)

chemin_parquet = Path("data/processed/licences_dep.parquet")

if chemin_parquet.exists():
    df_licences = pd.read_parquet(chemin_parquet)
else:
    df_licences = cl.construire_base_departementale()

print("Années disponibles :", sorted(df_licences["annee"].unique()))
print("Exemples de fédérations :", sorted(df_licences["nom_fed"].dropna().unique())[:15])

Années disponibles : [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Exemples de fédérations : ['F Nationale du Sport en Milieu Rural', 'F Nautique de Pêche Sportive  en Apnée', "F Sportive Educative de l'Enseignement Catholique (UGSEL)", 'F Sportive de la Police Nationale', 'F Sportive des ASPTT', 'F Sportive et Culturelle de France', 'F Sportive et Gymnique du Travail', 'F de Boxe Américaine et DA', 'F de Double Dutch', 'F de Voitures Radio Commandées', 'F des Arts Énergétiques et Martiaux Chinois', 'F des Clubs de la Défense', 'F des Sports et Loisirs Canins', 'F nationale du sport en milieu rural', 'F sportive et culturelle de France']


In [11]:
from src.processing import (
    preparer_serie_base_100,
    calculer_ecart_double_difference,
    calculer_croissance_departementale,
)
from src.viz import plot_effet_local_base100, plot_carte_croissance_dep

# Exemple : Tennis de table et les frères Lebrun (Hérault - 34)
CODE_FFTT = "124"

df_ping_100 = preparer_serie_base_100(
    df_licences,
    federation=CODE_FFTT,
    dep_cible="34",
    annee_ref=2019,
    label_cible="Hérault (34 - Fief Lebrun)",
    col_fed="code_fed",
)

ecart_did = calculer_ecart_double_difference(
    df_ping_100,
    label_cible="Hérault (34 - Fief Lebrun)",
    annee_avant=2016,
    annee_apres=2023,
)
print(f"Surplus de croissance dans l'Hérault (2019-2023) : {ecart_did:+.1f} points d'indice")

fig_courbe = plot_effet_local_base100(
    df_ping_100, "Tennis de table", "Frères Lebrun", annee_ref=2019, annee_evenement=2021
)
fig_courbe.show()

df_ping_carte = calculer_croissance_departementale(
    df_licences,
    federation=CODE_FFTT,
    annee_debut=2016,
    annee_fin=2023,
    col_fed="code_fed",
)
fig_carte = plot_carte_croissance_dep(df_ping_carte, "Tennis de table", annee_debut=2016, annee_fin=2023)
fig_carte.show()

KeyError: 2016

In [ ]:
import importlib
from pathlib import Path
import pandas as pd
import src.collect_licences as cl
from src.processing import (
    preparer_serie_base_100,
    calculer_ecart_double_difference,
    calculer_croissance_departementale,
)
from src.viz import plot_effet_local_base100, plot_carte_croissance_dep

importlib.reload(cl)

# 1. Chargement ou génération de la base départementale 2016-2025
chemin_complet = Path("data/processed/licences_dep_2016_2025.parquet")
if chemin_complet.exists():
    df_licences = pd.read_parquet(chemin_complet)
else:
    df_hist = pd.read_parquet("data/processed/licences_dep.parquet")
    df_licences = cl.enrichir_avec_2025(df_hist)

print("Années disponibles :", sorted(df_licences["annee"].unique()))

# 2. Vérifie les codes fédérations (ex. FFTT, Natation, Basket, Judo, Cyclisme)
mots_cles = ["tennis de table", "natation", "basket", "judo", "cyclisme"]
masque = df_licences["nom_fed"].str.lower().str.contains("|".join(mots_cles), na=False)
display(df_licences[masque][["code_fed", "nom_fed"]].drop_duplicates().sort_values("code_fed"))

Base 2016-2025 sauvegardée dans data/processed/licences_dep_2016_2025.parquet (87044 lignes).
Années disponibles : [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2025)]


,code_fed,nom_fed
3,105,Fédération française de basketball
9453,105,Fédération Française de basketball
18188,105,FF de basketball
26853,105,FF de Basketball
74287,105,Fédération Française de Basketball
6,108,Fédération française de cyclisme
9456,108,Fédération Française de cyclisme
18190,108,FF de cyclisme
26856,108,FF de Cyclisme
74290,108,Fédération Française de Cyclisme


In [ ]:
CODE_FFTT = "124"  # À ajuster selon le tableau ci-dessus

df_ping_100 = preparer_serie_base_100(
    df_licences,
    federation=CODE_FFTT,
    dep_cible="34",
    annee_ref=2019,
    label_cible="Hérault (34 - Fief Lebrun)",
    col_fed="code_fed",
)

ecart_did = calculer_ecart_double_difference(
    df_ping_100, label_cible="Hérault (34 - Fief Lebrun)", annee_avant=2019, annee_apres=2025
)
print(f"Surplus de croissance dans l'Hérault (2023-2025) : {ecart_did:+.1f} points d'indice")

fig_courbe = plot_effet_local_base100(df_ping_100, "Tennis de table", "Frères Lebrun", annee_ref=2019, annee_evenement=2024.5)
fig_courbe.show()

df_ping_carte = calculer_croissance_departementale(
    df_licences, federation=CODE_FFTT, annee_debut=2019, annee_fin=2025, col_fed="code_fed"
)
fig_carte = plot_carte_croissance_dep(df_ping_carte, "Tennis de table", annee_debut=2019, annee_fin=2025)
fig_carte.show()

Surplus de croissance dans l'Hérault (2023-2025) : +15.1 points d'indice


In [ ]:
CODE_FFTT = "124"  # À ajuster selon le tableau ci-dessus

df_ping_100 = preparer_serie_base_100(
    df_licences,
    federation=CODE_FFTT,
    dep_cible="34",
    annee_ref=2023,
    label_cible="Hérault (34 - Fief Lebrun)",
    col_fed="code_fed",
)

ecart_did = calculer_ecart_double_difference(
    df_ping_100, label_cible="Hérault (34 - Fief Lebrun)", annee_avant=2023, annee_apres=2025
)
print(f"Surplus de croissance dans l'Hérault (2023-2025) : {ecart_did:+.1f} points d'indice")

fig_courbe = plot_effet_local_base100(df_ping_100, "Tennis de table", "Frères Lebrun", annee_ref=2023, annee_evenement=2024.5)
# 2024 correspond a septembre 2023- juin 2024 donc pas encore l'effet des JO (que de l'anticipation, mais osef ici)
fig_courbe.show()

df_ping_carte = calculer_croissance_departementale(
    df_licences, federation=CODE_FFTT, annee_debut=2023, annee_fin=2025, col_fed="code_fed"
)
fig_carte = plot_carte_croissance_dep(df_ping_carte, "Tennis de table", annee_debut=2023, annee_fin=2025)
fig_carte.show()

Surplus de croissance dans l'Hérault (2023-2025) : -0.3 points d'indice


### Analyse de l'effet « Héros local » : choix de l'année de référence (2019 vs 2023)

Le choix de l'année de référence conditionne l'hypothèse testée et permet de décomposer la diffusion de l'effet « frères Lebrun » en deux temps :

* **Base 100 en 2019 (Référence principale : effet global du rôle modèle)**
  * **Justification :** 2019 correspond à la dernière saison normale avant la crise sanitaire et avant l'émergence médiatique des frères Lebrun. En l'absence de choc local, l'Hérault (34) aurait dû suivre la même trajectoire de reprise que le reste de la France.
  * **Résultats :** On observe un surplus de croissance net de **+15 points d'indice** dans l'Hérault en 2025 (indice 120 contre 105 au niveau national). Cet écart se creuse dès la période pré-olympique 2021-2023 (indice 96 contre 84), portée par leurs premiers titres nationaux et européens.

* **Base 100 en 2023 (Vérification : effet marginal des JO de Paris 2024)**
  * **Justification :** Permet d'isoler l'impact pur des Jeux Olympiques de Paris 2024 une fois les athlètes déjà connus localement.
  * **Résultats :** Entre 2023 et 2025, les courbes de l'Hérault et du reste de la France se superposent (+25 % sur l'ensemble du territoire). L'exposition olympique a transformé un engouement initialement régional en un phénomène national homogène.

#### Limites méthodologiques à retenir
1. **Tendances pré-événement (2016-2019) :** L'Hérault affichait une légère érosion avant 2019 (passant de l'indice 107 à 100) contre une stabilité nationale, ce qui renforce la portée du retournement observé après 2021.
2. **Amplitude du creux Covid en 2021 :** La chute plus marquée des licences héraultaises en 2021 (indice 50 contre 60 au niveau national) explique une partie de la vigueur du rebond initial, bien que le dépassement durable du seuil de 2019 confirme un véritable effet d'entraînement.

### GENRE

In [ ]:
import importlib
import src.processing as pr
import src.viz as vz

importlib.reload(pr)
importlib.reload(vz)

df_ping_genre = pr.analyser_effet_genre(df_licences, federation="124", annee_ref=2019, col_fed="code_fed")
display(df_ping_genre[["annee", "licences_f", "licences_h", "taux_feminisation", "indice_f", "indice_h"]])

fig_genre = vz.plot_effet_genre(df_ping_genre, "Tennis de table", "Frères Lebrun", annee_ref=2019)
fig_genre.show()

,annee,licences_f,licences_h,taux_feminisation,indice_f,indice_h
0,2016,38767,166897,18.849677,92.502804,98.425982
1,2017,38358,165983,18.771563,91.526880,97.886958
2,2018,40199,166444,19.453357,95.919731,98.158829
3,2019,41909,169566,19.817473,100.000000,100.000000
4,2020,24421,151489,13.882667,58.271493,89.339254
5,2021,16436,109742,13.026043,39.218306,64.719342
6,2022,23722,134391,15.003194,56.603594,79.255865
7,2023,26953,150460,15.192235,64.313155,88.732411
8,2025,32364,189898,14.561194,77.224463,111.990611


### TELLEMENT D HOMME 
+20K DE LICENCIES HOMME PAR RAPPORT A 2019
-7K PR LES FEMMES :/

### Effet JO
* amplifie l'ecart
Nuance méthodologique (limite à noter) : On observe une chute brutale des licences féminines dès 2020 (-48 % contre -20 % chez les hommes), liée à la fois à l'arrêt de la pratique loisir pendant la crise sanitaire et à une évolution des licences promotionnelles/scolaires dans les statistiques INJEP à partir de 2019-2020. L'engouement suscité par les frères Lebrun (deux modèles masculins) a néanmoins accéléré le rattrapage masculin bien plus fortement que le rattrapage féminin.

==> Il faudra comp avec la natation (LM) + judo/cyclisme avec les femmes

## AGE

In [ ]:
import importlib
import src.collect_licences as cl
import src.viz as vz

importlib.reload(cl)
importlib.reload(vz)

df_ping_ages = cl.construire_base_ages_2019_2025(code_fed_cible="124")

# Part de chaque tranche d'âge dans les nouveaux licenciés gagnés entre 2023 et 2025
piv_age = df_ping_ages.pivot(index="tranche_age", columns="annee", values="licences")
piv_age["gain_2023_2025"] = piv_age[2025] - piv_age[2023]
piv_age["part_du_gain (%)"] = (piv_age["gain_2023_2025"] / piv_age["gain_2023_2025"].sum()) * 100
display(piv_age)

fig_age = vz.plot_effet_age(df_ping_ages, "Tennis de table", "Frères Lebrun", annee_ref=2023)
fig_age.show()

annee,2019,2020,2021,2022,2023,2025,gain_2023_2025,part_du_gain (%)
tranche_age,,,,,,,,
10 à 14 ans,55959.0,47864.0,32160.0,41470.0,47406.0,60259.0,12853.0,28.658387
15 à 19 ans,15905.0,13333.0,10987.0,12345.0,12838.0,14250.0,1412.0,3.148342
20 ans et plus,97561.0,91816.0,70603.0,84583.0,95196.0,121262.0,26066.0,58.119468
Moins de 10 ans,41928.0,22792.0,12318.0,19715.0,21973.0,26491.0,4518.0,10.073803


### Analyse de l'effet Âge (Base 100 en 2023) : un engouement généralisé

Le recalibrage sur **2023** permet d'isoler l'impact des Jeux Olympiques de Paris 2024 et des frères Lebrun sans le biais du décrochage des licences scolaires en 2020. Entre 2023 et 2025, la fédération gagne **+44 849 licenciés** et toutes les classes d'âge affichent une croissance à deux chiffres :

* **Deux moteurs à quasi-égalité en rythme de croissance (+27 %) :**
  * Les **20 ans et plus** progressent de **+27,4 %** (indice 127,4) et représentent **58,1 %** du gain total (+26 066 licenciés), portés par les jeunes actifs et les pratiquants loisirs ou anciens joueurs revenant en club.
  * Les **10 à 14 ans** suivent exactement la même dynamique avec **+27,1 %** (indice 127,1), générant **28,7 %** des nouveaux inscrits (+12 853 licenciés). Cette tranche d'âge incarne le cœur de l'identification aux frères Lebrun.
* **Une reprise confirmée chez les plus jeunes (+20,6 %) :** Les **moins de 10 ans** atteignent l'indice **120,6** (+4 518 licenciés, soit 10,1 % de la hausse). Bien qu'ils restent en deçà de leur volume d'avant-Covid en raison de la baisse structurelle des licences scolaires depuis 2020, leur rebond post-JO est bien réel.
* **Un effet plus modéré chez les grands adolescents (+11,0 %) :** Les **15 à 19 ans** progressent plus lentement (indice 111,0, soit 3,1 % du gain total), une inertie habituelle liée aux contraintes des examens et des études supérieures.

**Conclusion intermédiaire :** L'effet « Paris 2024 et frères Lebrun » ne se limite pas au public enfantin : les **moins de 20 ans** représentent **41,9 %** de la croissance totale, mais la vague d'adhésions touche avec la même intensité relative les collégiens (10-14 ans) et les adultes.

#### MEDAILLES JO

In [ ]:
!pip install openpyxl


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
import importlib
import src.collect_jo as cjo

importlib.reload(cjo)

donnees_jo = cjo.charger_donnees_jo()

df_athletes_2024 = donnees_jo["athletes_paris_2024"]
df_medailles_2012_2014 = donnees_jo["medailles_2012_2014"]

display(df_athletes_2024.head(3))
display(df_medailles_2012_2014.head(3))

Fichier téléchargé : data/raw/athletes_paris_2024.csv (source : paris2024-athletes.csv)

--- athletes_paris_2024 (11110 lignes) ---
Colonnes : ['nom', 'Lien vers la source', 'Discipline', 'Date de naissance', 'Pays', 'Âge', 'Fonction', 'Taille (en cm)', 'Genre', 'Site de compétition', 'Latitude', 'Longitude', 'Nombre de compétitions', 'Date de début', 'Date de fin', 'Jours de compétitions', "Médailles d'or", "Médailles d'argent", 'Médailles de bronze', 'Record']
Fichier téléchargé : data/raw/medailles_2012_2014.xlsx (source : Liste des médailles obtenues par des sportifs français aux Jeux Olympiques et paralympiques de 2012 et de 2014 )

--- medailles_2012_2014 (179 lignes) ---
Colonnes : ['Fédération', 'Discipline', 'Epreuve', 'NOM', 'Prénom', 'Sexe', 'MEDAILLE(S)', 'Année', 'Olympique/paralympique']


,nom,Lien vers la source,Discipline,Date de naissance,Pays,Âge,Fonction,Taille (en cm),Genre,Site de compétition,Latitude,Longitude,Nombre de compétitions,Date de début,Date de fin,Jours de compétitions,Médailles d'or,Médailles d'argent,Médailles de bronze,Record
0,"""671""",https://olympics.com/fr/paris-2024/athlete/-67...,Breaking,19/10/05,République populaire de Chine,18,Athlète,NaN,Femme,La Concorde 1,48.866406,2.321195,6,09/08/24,09/08/24,1,0,0,1,non
1,AABERG IVERSEN Sarah,https://olympics.com/fr/paris-2024/athlete/sar...,Handball,10/04/90,Danemark,34,Athlète,177.0,Femme,multisite,NaN,NaN,8,25/07/24,10/08/24,8,0,0,1,non
2,AARDAHL Maren,https://olympics.com/fr/paris-2024/athlete/mar...,Handball,02/03/94,Norvège,30,Athlète,183.0,Femme,multisite,NaN,NaN,8,25/07/24,10/08/24,8,1,0,0,non


,Fédération,Discipline,Epreuve,NOM,Prénom,Sexe,MEDAILLE(S),Année,Olympique/paralympique
0,Athlétisme,Athlétisme,3000m steeple homme,MEKHISSI BENABBAD,Mahiedine,M,Argent,2012,Olympique
1,Athlétisme,Athlétisme,Saut à la perche homme,LAVILLENIE,Renaud,M,Or,2012,Olympique
2,BASKET BALL,Basket-ball,Tournoi féminin,YACOUBOU,Isabelle,F,Argent,2012,Olympique


In [ ]:
import importlib
import src.processing as pr
import src.viz as vz

importlib.reload(pr)
importlib.reload(vz)

# 1. Synthèse Paris 2024 et Londres 2012
df_fr_2024 = pr.synthetiser_paris_2024_france(df_athletes_2024)
df_fr_2012 = pr.synthetiser_medailles_2012_2014(df_medailles_2012_2014, annee_jo=2012)

# Vérifie s'il manque des codes fédérations pour les disciplines françaises médaillées
print("Disciplines 2024 sans code_fed associé :")
display(df_fr_2024[df_fr_2024["code_fed"].isna()][["Discipline", "nb_athletes", "total_medailles"]])

# 2. Croisement avec la croissance des licences 2023-2025
df_bilan_2024 = pr.croiser_medailles_et_croissance(df_licences, df_fr_2024, annee_debut=2023, annee_fin=2025)
display(df_bilan_2024[["nom_fed", "disciplines_jo", "total_medailles", "medailles_or", "croissance_pct", "gain_net"]])

fig_scatter = vz.plot_medailles_vs_croissance(df_bilan_2024)
fig_scatter.show()

Disciplines 2024 sans code_fed associé :


,Discipline,nb_athletes,total_medailles
33,Surf,4,2
6,Breaking,4,1
28,Pentathlon moderne,4,1
14,Escalade,7,0
31,Skateboard,7,0
17,Golf,4,0


,nom_fed,disciplines_jo,total_medailles,medailles_or,croissance_pct,gain_net
19,Fédération Française de Tennis de Table,Tennis de table,4,0,25.279433,44849.0
8,Fédération Française d'Escrime,Escrime,16,1,21.848394,11702.0
2,Fédération Française de Badminton,Badminton,0,0,18.723647,37789.0
15,Fédération Française de Lutte,Lutte,0,0,16.780774,4755.0
22,Fédération Française de Triathlon et Disciplin...,Triathlon,2,1,16.219802,9903.0
21,Fédération Française de Tir à l'Arc,Tir à l'arc,4,0,14.963637,11049.0
11,"Fédération Française d'Haltérophilie, Musculation",Handball,17,0,14.039881,2929.0
20,Fédération Française de Tir,Tir,1,0,13.310410,34064.0
12,Fédération Française de Handball,Haltérophilie,0,0,13.227630,70353.0
24,Fédération Française de Volley,"Volleyball, Volleyball de plage",12,12,12.178685,17999.0


### Analyse transversale : les médailles suffisent-elles à créer un « effet JO » ?

Le croisement entre le bilan français aux Jeux Olympiques de Paris 2024 et l'évolution des licences entre 2023 et 2025 montre qu'**il n'existe pas de corrélation linéaire directe entre le nombre de médailles obtenues et le dynamisme des inscriptions** :

* **Le Tennis de table en tête de toutes les disciplines (~ +25 %) :** Avec 4 médailles individuelles (Félix Lebrun en simple, Alexis et Félix Lebrun et Simon Gauzy par équipes), la FFTT enregistre la plus forte croissance relative de l'ensemble du mouvement olympique français. L'identification aux frères Lebrun et la facilité d'accès à la pratique ont maximisé la conversion médiatique en adhésions.
* **L'effet vitrine de l'Escrime (~ +22 % pour 16 médailles) :** Seule discipline à allier un très haut volume de podiums et une croissance supérieure à 20 %, portée par l'exposition exceptionnelle des épreuves au Grand Palais.
* **Les contraintes d'infrastructures et d'effets de base :**
  * **Le paradoxe de la Natation (~ 0 % malgré 12 médailles) :** Malgré les exploits historiques de Léon Marchand, les effectifs de la FFN stagnent, freinés par la saturation des bassins et le manque de créneaux disponibles dans les piscines municipales.
  * **L'inertie des grands sports collectifs et de combat :** Le Basketball (28 médailles, +6 %), le Judo (23 médailles, +12 %) et le Football (21 médailles, +7 %) affichent des progressions régulières mais plus contenues en pourcentage en raison de leur volume initial déjà très élevé.
  * **Les disciplines en recul :** Les Sports équestres (-7 % malgré 6 médailles) et le Canoë-kayak (-10 % malgré 3 médailles) subissent des freins économiques (coût de la pratique) ou une baisse des licences loisirs.

**Conclusion :** Le succès sportif est un catalyseur, mais il ne se traduit par un boom des licences que lorsque la discipline dispose d'un **rôle modèle incarné** et d'une **capacité d'accueil matérielle suffisante** dans les clubs.

#### 2012 VS 2024

In [ ]:
import importlib
import src.collect_licences as cl
import src.processing as pr
import src.viz as vz

importlib.reload(cl)
importlib.reload(pr)
importlib.reload(vz)

# 1. Série longue de la FFTT (code 124)
df_histo = cl.charger_historique_national()
fig_histo = vz.plot_historique_olympique(df_histo, code_fed="124", nom_sport="Tennis de table")
fig_histo.show()

# 2. Test placebo sur les JO de Londres 2012 (croissance des licences entre 2012 et 2014)
df_fr_2012 = pr.synthetiser_medailles_2012_2014(df_medailles_2012_2014, annee_jo=2012)
df_bilan_2012 = pr.croiser_medailles_et_croissance(df_histo, df_fr_2012, annee_debut=2012, annee_fin=2014)

display(
    df_bilan_2012[
        ["nom_fed", "disciplines_jo", "total_medailles", "medailles_or", "croissance_pct", "gain_net"]
    ].head(10)
)

,nom_fed,disciplines_jo,total_medailles,medailles_or,croissance_pct,gain_net
1,Fédération Française d'Aviron,Aviron,2,0,12.840680,4914.0
0,Fédération Française d'Athlétisme,Athlétisme,2,1,12.282535,29038.0
2,Fédération Française de Gymnastique,Gymnastique artistique,1,0,4.401301,12600.0
3,"Fédération Française d'Haltérophilie, Musculation",Handball,15,15,4.087854,2092.0
5,Fédération Française de Taekwondo et Disciplin...,Taekwondo,2,0,2.397084,1256.0
4,"Fédération Française de Judo, Jujitsu, Kendo e...",Judo,7,2,-0.976882,-5602.0
6,Fédération Française de Tennis,Tennis,4,0,-2.332100,-25917.0


### Mise en perspective historique (2002–2025) : un record absolu pour la FFTT

L'analyse de la série longue de l'INJEP depuis 2002 démontre que la dynamique observée après Paris 2024 dépasse largement les fluctuations habituelles des cycles olympiques :

* **Deux décennies de stabilité sous les 195 000 licenciés :** Entre 2002 et 2019, les effectifs de la Fédération Française de Tennis de Table sont restés remarquablement stables, évoluant entre 175 000 et 193 000 adhérents sans jamais franchir la barre symbolique des 200 000 licenciés lors des précédentes olympiades (Athènes, Pékin, Londres ou Rio).
* **Un changement d'échelle inédit en 2025 :** Après le point bas de 2021 lié à la crise sanitaire (~126 000 licenciés) et une phase de normalisation jusqu'en 2024 (~182 000), la fédération atteint un sommet historique de **plus de 222 000 licenciés en 2025**, dépassant de près de **+30 000 pratiquants** son précédent record des années 2010.
* **L'avantage des Jeux à domicile et d'un rôle modèle incarné :** La comparaison avec le cycle de Londres 2012 confirme que les médailles obtenues à l'étranger n'entraînaient pas de hausse mécanique des adhésions (le judo et le tennis avaient même reculé entre 2012 et 2014 malgré leurs podiums). L'effet Paris 2024 repose sur la rencontre entre l'organisation à domicile et l'incarnation médiatique des frères Lebrun.

### IMPACT JO EN FR


In [ ]:
import importlib
import src.collect_licences as cl
import src.viz as vz

importlib.reload(cl)
importlib.reload(vz)

df_histo = cl.charger_historique_national(forcer=True)

fig_global = vz.plot_historique_global_france(df_histo, par_type=False)
fig_global.show()

fig_global_type = vz.plot_historique_global_france(df_histo, par_type=True)
fig_global_type.show()

### Contexte macroéconomique : l'impact global des Jeux sur le sport français (2002–2025)

L'analyse de la série longue nettoyée de l'INJEP permet de replacer le rebond du tennis de table dans la dynamique d'ensemble du mouvement sportif français :

* **Un sommet historique à 17,5 millions de licences en 2025 :** Après avoir effacé dès 2023 le creux de la crise sanitaire de 2021 (12,8 millions de licences), le total national atteint un niveau inédit de **17,1 millions en 2024** puis **17,5 millions en 2025**[cite: 7].
* **Une dynamique concentrée sur les disciplines olympiques :** La décomposition par type de fédération révèle que les fédérations **olympiques** portent la quasi-totalité de cette hausse, franchissant pour la première fois la barre des **10 millions de licenciés** en 2024–2025 (+6 % à +7 % entre 2023 et 2025), tandis que les fédérations scolaires, affinitaires et non-olympiques stagnent[cite: 7].
* **Un « effet Lebrun » nettement supérieur à la tendance olympique :** Avec une hausse d'environ **+25 %** de ses effectifs entre 2023 et 2025, la FFTT surperforme de près de **18 points** la moyenne des fédérations olympiques, confirmant l'existence d'un effet spécifique lié à la médiatisation de Félix et Alexis Lebrun au-delà du seul effet d'entraînement général des Jeux à domicile.

In [ ]:
url_histo = "https://data.sports.gouv.fr/api/explore/v2.1/catalog/datasets/lics_histo/exports/csv?delimiter=%3B"
df_brut_histo = pd.read_csv(url_histo, sep=";", nrows=5, dtype=str)
display(df_brut_histo)

,code_fede_2025,code_fede_ref,federation,annee,evol_20ans,evol_10ans,evol_5ans,lics,lics_100_2002,lics_100_2016,fede_gp
0,101,101,Fédération Française d'Athlétisme,2005-01-01,1.01571191626698,0.183773652580905,0.100433455154063,167007,100.693368383608,55.3047262034069,1. Olympique
1,101,101,Fédération Française d'Athlétisme,2013-01-01,1.01571191626698,0.183773652580905,0.100433455154063,256459,154.62657590575,84.9269478369142,1. Olympique
2,101,101,Fédération Française d'Athlétisme,2022-01-01,1.01571191626698,0.183773652580905,0.100433455154063,302583,182.436074449677,100.201009351737,1. Olympique
3,101,101,Fédération Française d'Athlétisme,2023-01-01,1.01571191626698,0.183773652580905,0.100433455154063,307021,185.111873481372,101.670662569211,1. Olympique
4,102,102,Fédération Française d'Aviron,2002-01-01,0.222102802482736,0.00229180898653558,0.042853212271289,39771,100.0,86.3741991530025,1. Olympique


In [ ]:
# pour avoir les ref des sports

mots_cles = ["tennis de table", "natation", "basket", "judo", "cyclisme"]
masque = df_licences["nom_fed"].str.lower().str.contains("|".join(mots_cles), na=False)

correspondance = (
    df_licences[masque][["code_fed", "nom_fed", "annee"]]
    .groupby(["code_fed", "nom_fed"])["annee"]
    .agg(["min", "max", "count"])
    .reset_index()
)
display(correspondance)

,code_fed,nom_fed,min,max,count
0,105,FF de Basketball,2019,2023,505
1,105,FF de basketball,2018,2018,101
2,105,Fédération Française de Basketball,2025,2025,110
3,105,Fédération Française de basketball,2017,2017,101
4,105,Fédération française de basketball,2016,2016,101
5,108,FF de Cyclisme,2019,2023,505
6,108,FF de cyclisme,2018,2018,101
7,108,Fédération Française de Cyclisme,2025,2025,110
8,108,Fédération Française de cyclisme,2017,2017,101
9,108,Fédération française de cyclisme,2016,2016,101
